# Chapter 19: Fine-tuning pretrained Transformers with Hugging Face

Bonus notebook (not part of the book). It follows on from chapters 14 and 15: instead of training a text classifier from scratch, we fine-tune a pretrained DistilBERT model on the IMDB movie review dataset.

## Setup

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import numpy as np
import torch
from transformers import set_seed

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## Loading the IMDB dataset

In [ ]:
from datasets import load_dataset

imdb = load_dataset("imdb")
print(imdb)

In [ ]:
# Use smaller subsets so training finishes quickly on Colab's free GPU.
# Increase these numbers (or remove .select) for better accuracy.
train_ds = imdb["train"].shuffle(seed=42).select(range(4000))
test_ds = imdb["test"].shuffle(seed=42).select(range(1000))

print(train_ds[0]["text"][:300])
print("Label:", train_ds[0]["label"])

## Tokenizing the text

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)


train_tok = train_ds.map(tokenize, batched=True)
test_tok = test_ds.map(tokenize, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

# Pads each batch dynamically to the longest sequence in that batch.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Loading the pretrained model

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

## Defining the evaluation metrics

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

## Fine-tuning the model

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="distilbert-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluating the fine-tuned model

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
from sklearn.metrics import classification_report

pred_output = trainer.predict(test_tok)
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

print(classification_report(y_true, y_pred, target_names=["NEGATIVE", "POSITIVE"]))

## Running inference on new reviews

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

reviews = [
    "An absolutely brilliant film with a gripping story and amazing acting.",
    "Boring, predictable and way too long. I almost fell asleep.",
    "It had a few good moments, but overall it was a letdown.",
]

for review, pred in zip(reviews, classifier(reviews)):
    print(f"{pred['label']:<9} ({pred['score']:.3f})  {review}")

## Saving and reloading the model

In [ ]:
trainer.save_model("distilbert-imdb-final")
tokenizer.save_pretrained("distilbert-imdb-final")

reloaded = pipeline("text-classification", model="distilbert-imdb-final")
print(reloaded("One of the best movies I have seen this year!"))